# Brain Tumor Detection Project Notebook

This is the main notebook for the entire project. It keeps all project assets inside `F:\\brain tumor detection`, downloads and normalizes the dataset, prepares splits, trains models, evaluates them, exports metrics to CSV, and saves visualizations.

## What This Notebook Covers

1. Environment setup and project-local cache enforcement
2. Kaggle dataset download with live progress reporting
3. Dataset normalization into `data/raw/primary/<class>/...`
4. Dataset audit, CSV summaries, and visualizations
5. Split preparation
6. Baseline CNN training
7. TumorDetNet training
8. Evaluation, metrics CSV export, and saved comparison plots

In [1]:
from __future__ import annotations

import json
import logging
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'pyproject.toml').exists() else current_dir.parent
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise RuntimeError(f'Could not resolve project root from {current_dir}')

SRC_DIR = PROJECT_ROOT / 'src'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DOWNLOAD_DIR = RAW_DIR / 'downloads'
EXTRACT_DIR = DOWNLOAD_DIR / 'extracted'
PRIMARY_DIR = RAW_DIR / 'primary'
SECONDARY_DIR = RAW_DIR / 'secondary'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
CACHE_DIR = OUTPUTS_DIR / 'cache'
METRICS_DIR = OUTPUTS_DIR / 'metrics'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
MODELS_DIR = OUTPUTS_DIR / 'models'
DEVELOPER_ONLY_DIR = PROJECT_ROOT / 'developer-only'
KAGGLE_DIR = DEVELOPER_ONLY_DIR / 'kaggle'
KAGGLE_JSON = KAGGLE_DIR / 'kaggle.json'

for path in [RAW_DIR, DOWNLOAD_DIR, EXTRACT_DIR, PRIMARY_DIR, SECONDARY_DIR, PROCESSED_DIR, OUTPUTS_DIR, CACHE_DIR, METRICS_DIR, FIGURES_DIR, MODELS_DIR, KAGGLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ['BRAIN_TUMOR_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['XDG_CACHE_HOME'] = str(CACHE_DIR)
os.environ['TORCH_HOME'] = str(CACHE_DIR / 'torch')
os.environ['MPLCONFIGDIR'] = str(CACHE_DIR / 'matplotlib')
os.environ['HF_HOME'] = str(CACHE_DIR / 'huggingface')
os.environ['TRANSFORMERS_CACHE'] = str(CACHE_DIR / 'huggingface' / 'transformers')

if not KAGGLE_JSON.exists():
    raise FileNotFoundError(
        f'Kaggle credentials not found. Put kaggle.json at: {KAGGLE_JSON}'
    )
os.environ['KAGGLE_CONFIG_DIR'] = str(KAGGLE_DIR)

logger = logging.getLogger('brain_tumor_project_notebook')
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(name)s | %(message)s'))
logger.addHandler(handler)
logger.propagate = False
training_logger = logging.getLogger('training')
training_logger.setLevel(logging.INFO)
training_logger.handlers.clear()
training_logger.addHandler(handler)
training_logger.propagate = False

logger.info('Project root resolved to %s', PROJECT_ROOT)
logger.info('All project storage stays inside %s', PROJECT_ROOT)


2026-03-27 15:26:42,102 | INFO | brain_tumor_project_notebook | Project root resolved to F:\brain tumor detection
2026-03-27 15:26:42,103 | INFO | brain_tumor_project_notebook | All project storage stays inside F:\brain tumor detection


In [ ]:
from src.config.settings import load_settings
from src.data.dataset_audit import collect_image_records, summarize_records
from src.data.splits import load_splits
from src.evaluation.reporting import save_confusion_matrix, save_training_curves, write_json
from src.models.factory import build_model
from src.training.engine import compute_class_weights, evaluate_model, resolve_device, run_training
from src.utils.seed import set_global_seed
from src.utils.training_io import create_dataloaders, load_checkpoint

settings = load_settings(PROJECT_ROOT / 'src' / 'config' / 'default.yaml')
settings.data['paths']['root'] = str(PROJECT_ROOT)
settings.paths

{'root': 'F:\\brain tumor detection',
 'raw_data_dir': 'data/raw',
 'processed_data_dir': 'data/processed',
 'outputs_dir': 'outputs',
 'cache_dir': 'outputs/cache',
 'metrics_dir': 'outputs/metrics',
 'models_dir': 'outputs/models',
 'figures_dir': 'outputs/figures',
 'logs_dir': 'outputs/logs',
 'primary_dataset_dir': 'data/raw/primary',
 'primary_splits_csv': 'data/processed/primary_splits.csv'}

In [ ]:
def ensure_package(package_name: str, import_name: str | None = None) -> None:
    import_name = import_name or package_name
    try:
        __import__(import_name)
        logger.info('Package available: %s', package_name)
    except ImportError:
        logger.info('Installing package: %s', package_name)
        subprocess.run([sys.executable, '-m', 'pip', 'install', package_name], check=True)


def run_command(command: list[str], description: str) -> dict[str, object]:
    logger.info('Starting: %s', description)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding='utf-8',
        errors='replace',
        bufsize=1,
    )

    output_lines: list[str] = []
    epoch_progress = None
    last_epoch = 0
    assert process.stdout is not None
    for raw_line in process.stdout:
        line = raw_line.rstrip()
        if not line:
            continue
        output_lines.append(line)
        epoch_match = re.search(r'Epoch (\d+)/(\d+) \|', line)
        if epoch_match:
            current_epoch = int(epoch_match.group(1))
            total_epochs = int(epoch_match.group(2))
            if epoch_progress is None:
                epoch_progress = tqdm(total=total_epochs, desc=description, unit='epoch')
            if current_epoch > last_epoch:
                epoch_progress.update(current_epoch - last_epoch)
                last_epoch = current_epoch
            print(line)
            continue
        is_tqdm_update = '%|' in line or line.startswith('Epochs[')
        is_completed_tqdm = '100%|' in line
        if is_tqdm_update and not is_completed_tqdm:
            continue
        print(line)

    return_code = process.wait()
    if epoch_progress is not None:
        if last_epoch < epoch_progress.total:
            epoch_progress.update(epoch_progress.total - last_epoch)
        epoch_progress.close()
    if return_code != 0:
        raise RuntimeError(
            f'{description} failed with exit code {return_code}\n' + '\n'.join(output_lines[-50:])
        )
    logger.info('Completed: %s', description)
    return {'returncode': return_code, 'output_lines': output_lines}


def run_kaggle_download_with_progress() -> Path:
    archive_path = DOWNLOAD_DIR / 'brain-tumor-mri-dataset.zip'
    command = [
        sys.executable,
        '-m',
        'kaggle.cli',
        'datasets',
        'download',
        '-d',
        'masoudnickparvar/brain-tumor-mri-dataset',
        '-p',
        str(DOWNLOAD_DIR),
        '--force',
    ]

    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_lines: list[str] = []
    progress = tqdm(total=100, desc='Downloading dataset', unit='%')
    last_percent = 0

    assert process.stdout is not None
    for raw_line in process.stdout:
        line = raw_line.strip()
        if not line:
            continue
        output_lines.append(line)
        logger.info('%s', line)
        match = re.search(r'(\d+)%', line)
        if match:
            percent = int(match.group(1))
            if percent > last_percent:
                progress.update(percent - last_percent)
                last_percent = percent

    return_code = process.wait()
    if return_code == 0 and last_percent < 100:
        progress.update(100 - last_percent)
    progress.close()

    if return_code != 0:
        raise RuntimeError(
            'Kaggle download failed. Output:\n' + '\n'.join(output_lines)
        )

    if archive_path.exists():
        return archive_path

    zip_files = sorted(DOWNLOAD_DIR.glob('*.zip'))
    if len(zip_files) != 1:
        raise FileNotFoundError(
            f'Expected one zip file in {DOWNLOAD_DIR}, found {len(zip_files)}'
        )
    return zip_files[0]


def extract_archive(archive_path: Path) -> None:
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    logger.info('Extracting %s into %s', archive_path, EXTRACT_DIR)
    shutil.unpack_archive(str(archive_path), str(EXTRACT_DIR))


def infer_label(image_path: Path) -> str | None:
    ignored = {'training', 'testing', 'train', 'test', 'val', 'validation'}
    parent_name = image_path.parent.name
    if parent_name.lower() not in ignored:
        return parent_name
    for part in reversed(image_path.parts[:-1]):
        if part.lower() not in ignored:
            return part
    return None


def normalize_primary_dataset() -> pd.DataFrame:
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    image_paths = [
        path for path in EXTRACT_DIR.rglob('*')
        if path.is_file() and path.suffix.lower() in valid_extensions
    ]
    if not image_paths:
        raise FileNotFoundError(f'No images found under {EXTRACT_DIR}')

    for class_dir in [path for path in PRIMARY_DIR.iterdir() if path.is_dir()]:
        shutil.rmtree(class_dir)

    rows = []
    for image_path in tqdm(image_paths, desc='Normalizing dataset layout'):
        label = infer_label(image_path)
        if label is None:
            continue
        destination_dir = PRIMARY_DIR / label
        destination_dir.mkdir(parents=True, exist_ok=True)
        destination_path = destination_dir / f'{image_path.parent.name}_{image_path.name}'
        shutil.copy2(image_path, destination_path)
        rows.append({'label': label, 'path': str(destination_path)})

    normalized_df = pd.DataFrame(rows)
    if normalized_df.empty:
        raise RuntimeError('Dataset normalization produced no files.')

    normalized_csv = METRICS_DIR / 'normalized_dataset_inventory.csv'
    normalized_df.to_csv(normalized_csv, index=False)
    logger.info('Saved normalized dataset inventory to %s', normalized_csv)
    return normalized_df


def save_class_distribution_plot(class_counts: pd.Series, output_path: Path, title: str) -> None:
    plt.figure(figsize=(10, 6))
    sns.barplot(x=class_counts.index, y=class_counts.values, hue=class_counts.index, legend=False, palette='Blues_d')
    plt.title(title)
    plt.xlabel('Class')
    plt.ylabel('Images')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()
    logger.info('Saved figure to %s', output_path)


def save_sample_grid(output_path: Path, images_per_class: int = 4) -> None:
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    class_dirs = sorted(path for path in PRIMARY_DIR.iterdir() if path.is_dir())
    if not class_dirs:
        raise RuntimeError('No class folders found in the normalized primary dataset.')

    num_rows = len(class_dirs)
    num_cols = images_per_class
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(4 * num_cols, 3 * num_rows))
    if num_rows == 1:
        axes = [axes]

    for row_index, class_dir in enumerate(class_dirs):
        image_paths = [path for path in sorted(class_dir.glob('*')) if path.suffix.lower() in valid_extensions][:images_per_class]
        for col_index in range(num_cols):
            ax = axes[row_index][col_index] if num_rows > 1 else axes[col_index]
            ax.axis('off')
            if col_index < len(image_paths):
                with Image.open(image_paths[col_index]) as image:
                    ax.imshow(image)
                ax.set_title(class_dir.name)

    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()
    logger.info('Saved figure to %s', output_path)


def flatten_metrics(results: dict, split_name: str) -> dict[str, object]:
    metrics = results[split_name]
    return {
        'model': results['model'],
        'split': split_name,
        'accuracy': metrics['accuracy'],
        'precision': metrics['precision'],
        'recall': metrics['recall'],
        'f1_score': metrics['f1_score'],
        'roc_auc': metrics['roc_auc'],
        'loss': metrics['loss'],
    }


def normalized_dataset_exists() -> bool:
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    class_dirs = [path for path in PRIMARY_DIR.iterdir() if path.is_dir()]
    if not class_dirs:
        return False
    for class_dir in class_dirs:
        if any(path.is_file() and path.suffix.lower() in valid_extensions for path in class_dir.rglob('*')):
            return True
    return False


def train_model_in_notebook(model_name: str) -> dict[str, object]:
    set_global_seed(int(settings.project['seed']))
    split_df = load_splits(PROCESSED_DIR / 'primary_splits.csv')
    train_loader, val_loader, test_loader, class_names, train_labels = create_dataloaders(split_df, settings.dataset)

    model = build_model(
        model_name=model_name,
        num_classes=len(class_names),
        channels=int(settings.dataset['channels']),
        model_cfg=settings.models[model_name],
    )

    class_weights = None
    if bool(settings.training['use_weighted_loss']):
        class_weights = compute_class_weights(train_labels)

    total_epochs = int(settings.training['epochs'])
    epoch_bar = tqdm(total=total_epochs, desc=f'{model_name} epochs', unit='epoch')
    phase_bar = tqdm(total=1, desc=f'{model_name} phase', unit='batch', leave=False)
    last_epoch = 0
    device = resolve_device(settings.training['device'])
    logger.info('Notebook training %s on device %s', model_name, device)

    def progress_callback(event: dict[str, object]) -> None:
        nonlocal last_epoch
        event_name = str(event['event'])
        if event_name == 'epoch_start':
            current_epoch = int(event['epoch'])
            if current_epoch > last_epoch:
                epoch_bar.update(current_epoch - last_epoch)
                last_epoch = current_epoch
        elif event_name == 'phase_start':
            total_batches = int(event['total_batches'])
            phase = str(event['phase'])
            epoch_number = int(event['epoch'])
            phase_bar.reset(total=total_batches)
            phase_bar.set_description(f'{model_name} {phase} e{epoch_number}/{total_epochs}')
            phase_bar.refresh()
        elif event_name == 'phase_progress':
            batch_index = int(event['batch'])
            increment = batch_index - phase_bar.n
            if increment > 0:
                phase_bar.update(increment)
        elif event_name == 'phase_end':
            if phase_bar.total is not None and phase_bar.n < phase_bar.total:
                phase_bar.update(phase_bar.total - phase_bar.n)
            phase_bar.refresh()
        elif event_name == 'epoch_end':
            train_metrics = event['train_metrics']
            val_metrics = event['val_metrics']
            test_metrics = event['test_metrics']
            metric_line = (
                f"Epoch {event['epoch']}/{event['total_epochs']} | "
                f"train loss {event['train_loss']:.4f} acc {event['train_accuracy']:.4f} "
                f"prec {train_metrics['precision']:.4f} rec {train_metrics['recall']:.4f} "
                f"f1 {train_metrics['f1_score']:.4f} auc {train_metrics['roc_auc']:.4f} | "
                f"val loss {event['val_loss']:.4f} acc {event['val_accuracy']:.4f} "
                f"prec {val_metrics['precision']:.4f} rec {val_metrics['recall']:.4f} "
                f"f1 {val_metrics['f1_score']:.4f} auc {val_metrics['roc_auc']:.4f} | "
                f"test loss {event['test_loss']:.4f} acc {event['test_accuracy']:.4f} "
                f"prec {test_metrics['precision']:.4f} rec {test_metrics['recall']:.4f} "
                f"f1 {test_metrics['f1_score']:.4f} auc {test_metrics['roc_auc']:.4f}"
            )
            tqdm.write(metric_line)

    try:
        artifacts = run_training(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader,
            training_cfg=settings.training,
            evaluation_cfg=settings.evaluation,
            class_names=class_names,
            checkpoint_dir=MODELS_DIR,
            checkpoint_stem=model_name,
            class_weights=class_weights,
            progress_callback=progress_callback,
            show_epoch_progress=False,
            show_batch_progress=False,
        )
    finally:
        if last_epoch < total_epochs:
            epoch_bar.update(total_epochs - last_epoch)
        epoch_bar.close()
        phase_bar.close()

    curves_path = FIGURES_DIR / f'{model_name}_training_curves.png'
    save_training_curves(artifacts.history, curves_path, title=model_name)

    history_df = pd.DataFrame(artifacts.history)
    history_df.insert(0, 'epoch', range(1, len(history_df) + 1))
    history_csv = METRICS_DIR / f'{model_name}_history.csv'
    history_df.to_csv(history_csv, index=False)

    model = model.to(device)
    load_checkpoint(model, artifacts.best_checkpoint_path, device)

    val_metrics = evaluate_model(model, val_loader, device, settings.evaluation['average'], class_names)
    test_metrics = evaluate_model(model, test_loader, device, settings.evaluation['average'], class_names)

    result = {
        'model': model_name,
        'class_names': class_names,
        'best_checkpoint': str(artifacts.best_checkpoint_path),
        'history': artifacts.history,
        'validation_metrics': val_metrics,
        'test_metrics': test_metrics,
    }

    result_path = METRICS_DIR / f'{model_name}_results.json'
    write_json(result, result_path)
    logger.info('Saved training history CSV to %s', history_csv)
    logger.info('Saved training results JSON to %s', result_path)
    return result


def evaluate_checkpoint_in_notebook(model_name: str, checkpoint_path: Path) -> dict[str, object]:
    split_df = load_splits(PROCESSED_DIR / 'primary_splits.csv')
    _, _, test_loader, class_names, _ = create_dataloaders(split_df, settings.dataset)
    device = resolve_device(settings.training['device'])
    model = build_model(
        model_name=model_name,
        num_classes=len(class_names),
        channels=int(settings.dataset['channels']),
        model_cfg=settings.models[model_name],
    )
    model = model.to(device)
    load_checkpoint(model, checkpoint_path, device)

    test_metrics = evaluate_model(model, test_loader, device, settings.evaluation['average'], class_names)
    metrics_path = METRICS_DIR / f'{model_name}_evaluation.json'
    confusion_path = FIGURES_DIR / f'{model_name}_confusion_matrix.png'
    write_json(test_metrics, metrics_path)
    save_confusion_matrix(test_metrics['confusion_matrix'], class_names, confusion_path, title=f'{model_name} Test Confusion Matrix')
    logger.info('Saved evaluation JSON to %s', metrics_path)
    logger.info('Saved confusion matrix figure to %s', confusion_path)
    return test_metrics


In [ ]:
ensure_package('kaggle', 'kaggle')
if normalized_dataset_exists():
    logger.info('Normalized dataset already exists in %s. Skipping download and extraction.', PRIMARY_DIR)
    normalized_df = pd.DataFrame([
        {'label': path.parent.name, 'path': str(path)}
        for path in PRIMARY_DIR.rglob('*')
        if path.is_file() and path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}
    ])
else:
    archive_path = run_kaggle_download_with_progress()
    logger.info('Downloaded archive: %s', archive_path)
    extract_archive(archive_path)
    normalized_df = normalize_primary_dataset()
normalized_df.head()

2026-03-27 15:26:42,166 | INFO | brain_tumor_project_notebook | Package available: kaggle
2026-03-27 15:26:42,168 | INFO | brain_tumor_project_notebook | Normalized dataset already exists in F:\brain tumor detection\data\raw\primary. Skipping download and extraction.


,label,path
0,glioma,F:\brain tumor detection\data\raw\primary\glio...
1,glioma,F:\brain tumor detection\data\raw\primary\glio...
2,glioma,F:\brain tumor detection\data\raw\primary\glio...
3,glioma,F:\brain tumor detection\data\raw\primary\glio...
4,glioma,F:\brain tumor detection\data\raw\primary\glio...


In [ ]:
valid_extensions = tuple(ext.lower() for ext in settings.dataset['valid_extensions'])
records = collect_image_records(PRIMARY_DIR, valid_extensions)
summary = summarize_records(records)
summary_path = METRICS_DIR / 'primary_dataset_summary.json'
with summary_path.open('w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)
logger.info('Saved dataset summary to %s', summary_path)

audit_df = pd.DataFrame(records)
audit_csv = METRICS_DIR / 'primary_dataset_audit.csv'
audit_df.to_csv(audit_csv, index=False)
logger.info('Saved dataset audit CSV to %s', audit_csv)

class_counts = audit_df['label'].value_counts().sort_index()
class_count_csv = METRICS_DIR / 'primary_class_counts.csv'
class_counts.rename_axis('label').reset_index(name='count').to_csv(class_count_csv, index=False)
logger.info('Saved class counts CSV to %s', class_count_csv)

save_class_distribution_plot(class_counts, FIGURES_DIR / 'primary_class_distribution.png', 'Primary Dataset Class Distribution')
save_sample_grid(FIGURES_DIR / 'primary_sample_grid.png')
summary

2026-03-27 15:26:45,000 | INFO | brain_tumor_project_notebook | Saved dataset summary to F:\brain tumor detection\outputs\metrics\primary_dataset_summary.json
2026-03-27 15:26:45,016 | INFO | brain_tumor_project_notebook | Saved dataset audit CSV to F:\brain tumor detection\outputs\metrics\primary_dataset_audit.csv
2026-03-27 15:26:45,019 | INFO | brain_tumor_project_notebook | Saved class counts CSV to F:\brain tumor detection\outputs\metrics\primary_class_counts.csv
2026-03-27 15:26:45,120 | INFO | brain_tumor_project_notebook | Saved figure to F:\brain tumor detection\outputs\figures\primary_class_distribution.png
2026-03-27 15:26:46,174 | INFO | brain_tumor_project_notebook | Saved figure to F:\brain tumor detection\outputs\figures\primary_sample_grid.png


{'num_images': 7200,
 'num_classes': 4,
 'classes': {'glioma': 1800,
  'meningioma': 1800,
  'notumor': 1800,
  'pituitary': 1800},
 'image_width_range': [150, 1375],
 'image_height_range': [167, 1446]}

In [ ]:
run_command(
    [sys.executable, 'scripts/prepare_dataset.py', '--config', 'src/config/default.yaml'],
    'prepare dataset splits'
)

splits_df = load_splits(PROCESSED_DIR / 'primary_splits.csv')
split_counts = splits_df.groupby(['split', 'label']).size().reset_index(name='count')
split_counts_csv = METRICS_DIR / 'split_class_counts.csv'
split_counts.to_csv(split_counts_csv, index=False)
logger.info('Saved split counts CSV to %s', split_counts_csv)

plt.figure(figsize=(10, 6))
sns.barplot(data=split_counts, x='label', y='count', hue='split')
plt.title('Class Distribution by Split')
plt.xlabel('Class')
plt.ylabel('Images')
plt.xticks(rotation=15)
plt.tight_layout()
split_plot_path = FIGURES_DIR / 'split_class_distribution.png'
plt.savefig(split_plot_path)
plt.close()
logger.info('Saved split distribution plot to %s', split_plot_path)
split_counts

NameError: name 'run_command' is not defined

In [ ]:
baseline_results = train_model_in_notebook('baseline_cnn')
baseline_results['test_metrics']

baseline_cnn epochs:   0%|          | 0/15 [00:00<?, ?epoch/s]

baseline_cnn phase:   0%|          | 0/1 [00:00<?, ?batch/s]

2026-03-27 15:26:52,100 | INFO | brain_tumor_project_notebook | Notebook training baseline_cnn on device cuda
2026-03-27 15:26:52,102 | INFO | training | Training baseline_cnn for up to 15 epochs on device cuda
2026-03-27 15:27:37,427 | INFO | training | Epoch 1/15 | train loss 1.2142 acc 0.4381 prec 0.4272 rec 0.4381 f1 0.4239 auc 0.6885 | val loss 1.1138 acc 0.4981 prec 0.4273 rec 0.4981 f1 0.4392 auc 0.7771 | test loss 1.0860 acc 0.5157 prec 0.4454 rec 0.5157 f1 0.4558 auc 0.7904
Epoch 1/15 | train loss 1.2142 acc 0.4381 prec 0.4272 rec 0.4381 f1 0.4239 auc 0.6885 | val loss 1.1138 acc 0.4981 prec 0.4273 rec 0.4981 f1 0.4392 auc 0.7771 | test loss 1.0860 acc 0.5157 prec 0.4454 rec 0.5157 f1 0.4558 auc 0.7904
2026-03-27 15:28:23,437 | INFO | training | Epoch 2/15 | train loss 1.1319 acc 0.5071 prec 0.5009 rec 0.5071 f1 0.4970 auc 0.7486 | val loss 1.1211 acc 0.5000 prec 0.6692 rec 0.5000 f1 0.4503 auc 0.7934 | test loss 1.0991 acc 0.5185 prec 0.7346 rec 0.5185 f1 0.4693 auc 0.8055
Ep

{'accuracy': 0.8333333333333334,
 'precision': 0.8466989472536249,
 'recall': 0.8333333333333334,
 'f1_score': 0.8346360552779049,
 'confusion_matrix': [[197, 56, 11, 6],
  [4, 218, 18, 30],
  [0, 15, 245, 10],
  [3, 25, 2, 240]],
 'roc_auc': 0.9586716963877459,
 'class_metrics': {'glioma': {'precision': 0.9656862745098039,
   'recall': 0.7296296296296296,
   'f1_score': 0.8312236286919831},
  'meningioma': {'precision': 0.6942675159235668,
   'recall': 0.8074074074074075,
   'f1_score': 0.7465753424657534},
  'notumor': {'precision': 0.8876811594202898,
   'recall': 0.9074074074074074,
   'f1_score': 0.8974358974358975},
  'pituitary': {'precision': 0.8391608391608392,
   'recall': 0.8888888888888888,
   'f1_score': 0.8633093525179856}},
 'loss': 0.45585599221565104,
 'accuracy_from_epoch': 0.8333333333333334}

In [ ]:
tumordetnet_results = train_model_in_notebook('tumordetnet')
tumordetnet_results['test_metrics']

tumordetnet epochs:   0%|          | 0/15 [00:00<?, ?epoch/s]

tumordetnet phase:   0%|          | 0/1 [00:00<?, ?batch/s]

2026-03-27 15:38:30,977 | INFO | brain_tumor_project_notebook | Notebook training tumordetnet on device cuda
2026-03-27 15:38:30,988 | INFO | training | Training tumordetnet for up to 15 epochs on device cuda
2026-03-27 15:39:22,293 | INFO | training | Epoch 1/15 | train loss 0.8977 acc 0.6276 prec 0.6203 rec 0.6276 f1 0.6211 auc 0.8504 | val loss 0.9936 acc 0.5389 prec 0.7029 rec 0.5389 f1 0.4819 auc 0.8917 | test loss 0.9715 acc 0.5546 prec 0.7253 rec 0.5546 f1 0.4918 auc 0.8979
Epoch 1/15 | train loss 0.8977 acc 0.6276 prec 0.6203 rec 0.6276 f1 0.6211 auc 0.8504 | val loss 0.9936 acc 0.5389 prec 0.7029 rec 0.5389 f1 0.4819 auc 0.8917 | test loss 0.9715 acc 0.5546 prec 0.7253 rec 0.5546 f1 0.4918 auc 0.8979
2026-03-27 15:40:13,495 | INFO | training | Epoch 2/15 | train loss 0.7639 acc 0.6865 prec 0.6792 rec 0.6865 f1 0.6810 auc 0.8887 | val loss 1.2008 acc 0.4963 prec 0.7092 rec 0.4963 f1 0.4539 auc 0.8671 | test loss 1.1912 acc 0.4944 prec 0.6881 rec 0.4944 f1 0.4463 auc 0.8687
Epoc

{'accuracy': 0.8972222222222223,
 'precision': 0.9017764741948949,
 'recall': 0.8972222222222223,
 'f1_score': 0.8968646945508166,
 'confusion_matrix': [[219, 41, 8, 2],
  [2, 231, 17, 20],
  [0, 6, 262, 2],
  [2, 5, 6, 257]],
 'roc_auc': 0.9838054412437128,
 'class_metrics': {'glioma': {'precision': 0.9820627802690582,
   'recall': 0.8111111111111111,
   'f1_score': 0.8884381338742393},
  'meningioma': {'precision': 0.8162544169611308,
   'recall': 0.8555555555555555,
   'f1_score': 0.8354430379746836},
  'notumor': {'precision': 0.89419795221843,
   'recall': 0.9703703703703703,
   'f1_score': 0.9307282415630551},
  'pituitary': {'precision': 0.9145907473309609,
   'recall': 0.9518518518518518,
   'f1_score': 0.9328493647912885}},
 'loss': 0.29674305463278733,
 'accuracy_from_epoch': 0.8972222222222223}

In [ ]:
baseline_eval = evaluate_checkpoint_in_notebook('baseline_cnn', MODELS_DIR / 'baseline_cnn_best.pt')
tumordetnet_eval = evaluate_checkpoint_in_notebook('tumordetnet', MODELS_DIR / 'tumordetnet_best.pt')
{'baseline_cnn': baseline_eval['accuracy'], 'tumordetnet': tumordetnet_eval['accuracy']}

2026-03-27 15:51:32,077 | INFO | brain_tumor_project_notebook | Saved evaluation JSON to F:\brain tumor detection\outputs\metrics\baseline_cnn_evaluation.json
2026-03-27 15:51:32,078 | INFO | brain_tumor_project_notebook | Saved confusion matrix figure to F:\brain tumor detection\outputs\figures\baseline_cnn_confusion_matrix.png
2026-03-27 15:51:38,254 | INFO | brain_tumor_project_notebook | Saved evaluation JSON to F:\brain tumor detection\outputs\metrics\tumordetnet_evaluation.json
2026-03-27 15:51:38,254 | INFO | brain_tumor_project_notebook | Saved confusion matrix figure to F:\brain tumor detection\outputs\figures\tumordetnet_confusion_matrix.png


{'baseline_cnn': 0.8333333333333334, 'tumordetnet': 0.8972222222222223}

In [ ]:
comparison_rows = [
    flatten_metrics(baseline_results, 'validation_metrics'),
    flatten_metrics(baseline_results, 'test_metrics'),
    flatten_metrics(tumordetnet_results, 'validation_metrics'),
    flatten_metrics(tumordetnet_results, 'test_metrics'),
]
comparison_df = pd.DataFrame(comparison_rows)
comparison_csv = METRICS_DIR / 'model_metrics_comparison.csv'
comparison_df.to_csv(comparison_csv, index=False)
logger.info('Saved comparison CSV to %s', comparison_csv)

metric_names = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
for metric_name in metric_names:
    plt.figure(figsize=(8, 5))
    metric_df = comparison_df[comparison_df['split'] == 'test_metrics']
    sns.barplot(data=metric_df, x='model', y=metric_name, hue='model', legend=False, palette='deep')
    plt.title(f'Test {metric_name.replace("_", " ").title()} Comparison')
    plt.ylim(0, 1)
    plt.tight_layout()
    output_path = FIGURES_DIR / f'test_{metric_name}_comparison.png'
    plt.savefig(output_path)
    plt.close()
    logger.info('Saved comparison figure to %s', output_path)

comparison_df

2026-03-27 15:51:38,270 | INFO | brain_tumor_project_notebook | Saved comparison CSV to F:\brain tumor detection\outputs\metrics\model_metrics_comparison.csv
2026-03-27 15:51:38,346 | INFO | brain_tumor_project_notebook | Saved comparison figure to F:\brain tumor detection\outputs\figures\test_accuracy_comparison.png
2026-03-27 15:51:38,421 | INFO | brain_tumor_project_notebook | Saved comparison figure to F:\brain tumor detection\outputs\figures\test_precision_comparison.png
2026-03-27 15:51:38,515 | INFO | brain_tumor_project_notebook | Saved comparison figure to F:\brain tumor detection\outputs\figures\test_recall_comparison.png
2026-03-27 15:51:38,591 | INFO | brain_tumor_project_notebook | Saved comparison figure to F:\brain tumor detection\outputs\figures\test_f1_score_comparison.png
2026-03-27 15:51:38,666 | INFO | brain_tumor_project_notebook | Saved comparison figure to F:\brain tumor detection\outputs\figures\test_roc_auc_comparison.png


,model,split,accuracy,precision,recall,f1_score,roc_auc,loss
0,baseline_cnn,validation_metrics,0.813889,0.826519,0.813889,0.813998,0.960269,0.459942
1,baseline_cnn,test_metrics,0.833333,0.846699,0.833333,0.834636,0.958672,0.455856
2,tumordetnet,validation_metrics,0.914815,0.918282,0.914815,0.914819,0.985400,0.279982
3,tumordetnet,test_metrics,0.897222,0.901776,0.897222,0.896865,0.983805,0.296743


## Outputs Produced by This Notebook

- Dataset summaries and inventories in `outputs/metrics/`
- Split CSVs and model metrics CSVs in `outputs/metrics/`
- Training curves, confusion matrices, sample grids, and comparison plots in `outputs/figures/`
- Best and latest model checkpoints in `outputs/models/`

The secondary dataset robustness stage is still not implemented in the notebook or scripts.